In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.insert(0, "../helpers/"); from run_parrellel import run_parallel

import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit

Inputs

In [2]:
features = pd.read_parquet(variables.FEATURES_UNIQUE_DATA_PATH)
targets = pd.read_parquet(variables.AGG_TARGET_DATASET_PATH)
features_ranked_ordered_unique = pd.read_parquet(variables.FEATURES_RANKED_ORDERED_UNIQUE_PATH)

In [3]:
print(features.shape)
print(targets.shape)
print(features_ranked_ordered_unique.shape)

(200, 71)
(736417, 96)
(71, 96)


Subset sample

In [4]:
# Randomly sample index
seed = np.random.default_rng(42)
index = seed.choice(len(features), size=variables.FEATURE_SELECTION_SUBSAMPLE_AMOUNT, replace=False)
index.sort()

features = features.iloc[index]
targets = targets.iloc[index]

In [5]:
# Split feature data into x parts of train / validation parts
multiple_train_validation_subsets_index_ranges = TimeSeriesSplit(n_splits=3).split(features)
multiple_train_validation_subsets_index_ranges = list(multiple_train_validation_subsets_index_ranges)

In [6]:
def generate_feature_size_search_space():
    max_num_of_features = features.shape[1]
    num_of_features_to_test_per_horizon = np.linspace(start=10, stop=max_num_of_features, num=10).round(0)
    display(num_of_features_to_test_per_horizon)
    return num_of_features_to_test_per_horizon

num_of_features_to_test_per_horizon = generate_feature_size_search_space()

array([10., 17., 24., 30., 37., 44., 51., 57., 64., 71.])

In [7]:
train_params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.1,
    "num_leaves": 31,
    "min_child_samples": 20,
    "bagging_fraction": 0.5,
    "bagging_freq": 1,
    "force_col_wise": True,
    "random_state": 42,
    "n_jobs": 1,
    "num_threads": 1,
    "verbose": -1,
}

Core logic 

Train a LGB model per horizon, per top number of features, per subset fold. Extract MAE of each model, which determines best number of top features for each horizon

models = (CV folds=3)  *  (horizons=96) * (top feature n search space = 10)

models = 2880

In [8]:
features_ranked_ordered_unique[:10]

,target_h1,target_h2,target_h3,target_h4,target_h5,target_h6,target_h7,target_h8,target_h9,target_h10,...,target_h87,target_h88,target_h89,target_h90,target_h91,target_h92,target_h93,target_h94,target_h95,target_h96
feature,,,,,,,,,,,,,,,,,,,,,
nsw_price_asinh_lag_1,4,4,3,5,8,1,1,5,3,1,...,91,92,15,50,17,34,31,67,30,58
nsw_price_asinh_lag_2,11,9,7,9,5,7,7,9,10,3,...,58,38,40,82,108,51,38,167,86,131
nsw_price_asinh_lag_4,13,13,12,13,13,12,13,12,11,8,...,74,30,24,23,30,16,43,32,45,40
nsw_price_asinh_lag_12,30,30,26,31,21,30,29,21,18,27,...,22,42,46,13,55,60,47,22,14,27
nsw_price_asinh_lag_48,101,125,87,114,57,101,130,103,109,127,...,9,12,19,36,19,24,27,15,10,10
nsw_price_asinh_lag_96,317,236,216,258,244,303,195,158,198,223,...,75,57,160,130,80,71,71,229,113,166
nsw_price_asinh_lag_336,200,278,170,173,175,182,182,223,168,237,...,96,64,216,232,200,275,212,307,350,314
nsw_price_asinh_lag_337,217,123,144,201,211,209,155,147,219,195,...,70,225,252,193,183,194,244,290,278,293
nsw_price_asinh_rmean_48,37,29,18,26,23,27,27,24,29,19,...,43,27,23,53,26,47,109,43,60,30


In [9]:
def find_num_best_features_for_horizon(horizon_index):
    # Select the feature ranking scores for this horizon only
    top_feature_for_this_horizon = features_ranked_ordered_unique.iloc[:, horizon_index]
    # Explicitly sort by score high to low and use feature names from the index
    top_feature_for_this_horizon = top_feature_for_this_horizon.sort_values(ascending=False)
    
    # Filter the underlying features dataset by the ordered top features
    all_features_for_this_horizon = features.loc[:, top_feature_for_this_horizon.index]

    MAE_per_fold_per_top_feature_amount = []

    # For each split (which as a train and valid component)
    for train_index, valid_index in multiple_train_validation_subsets_index_ranges:
        subset_x_features_train = all_features_for_this_horizon.iloc[train_index]
        subset_x_features_valid = all_features_for_this_horizon.iloc[valid_index]
        subset_x_targets_train = targets.iloc[train_index, horizon_index]
        subset_x_targets_valid = targets.iloc[valid_index, horizon_index]

        # For each number of max features to test 
        for top_n_features in num_of_features_to_test_per_horizon.astype(int):
            
            
            features_for_this_horizon_top = subset_x_features_train.iloc[:, :top_n_features]
            valid_features_for_this_horizon_top = subset_x_features_valid.iloc[:, :top_n_features]
            train_data = lgb.Dataset(features_for_this_horizon_top, label=subset_x_targets_train, free_raw_data=False)
            model = lgb.train(train_params, train_data, num_boost_round=200)

            y_pred = model.predict(valid_features_for_this_horizon_top)
            average_prediction_error = np.mean(np.abs(subset_x_targets_valid - y_pred))

            MAE_per_fold_per_top_feature_amount.append((top_n_features, average_prediction_error))

            del train_data, model, y_pred

        del subset_x_features_train, subset_x_features_valid, subset_x_targets_train, subset_x_targets_valid

    return horizon_index, MAE_per_fold_per_top_feature_amount

results = run_parallel(
    function=find_num_best_features_for_horizon,
    data=list(range(variables.HORIZON_COUNT)),
)

os=windows cpu_count=8 max_assigned_workers=1


Processing..:   0%|          | 0/96 [00:00<?, ?item/s]

Processing..: 100%|██████████| 96/96 [01:07<00:00,  1.43item/s]


Clean up

In [10]:
model_results = pd.DataFrame(results, columns=["horizon", "maes"]).explode("maes").reset_index(drop=True)
model_results[["n_features", "mae"]] = pd.DataFrame(model_results.pop("maes").tolist(), index=model_results.index)
model_results_best = model_results.loc[model_results.groupby("horizon")["mae"].idxmin()].reset_index(drop=True)

In [11]:
# Create a dataframe mapping table for included features for each horizon

horizon_columns = [f"h{i + 1}" for i in range(variables.HORIZON_COUNT)]
feature_horizon_matrix = pd.DataFrame(False, index=features_ranked_ordered_unique.index, columns=horizon_columns)

for _, row in model_results_best.iterrows():
    horizon_index = int(row["horizon"])
    n_features = int(row["n_features"])
    selected = features_ranked_ordered_unique.iloc[:, horizon_index].sort_values(ascending=False).index[:n_features]
    feature_horizon_matrix.loc[selected, f"h{horizon_index + 1}"] = True

feature_horizon_matrix = feature_horizon_matrix.reset_index().rename(columns={"index": "feature"})


View

In [12]:
display(model_results_best)

,horizon,n_features,mae
0,0,57,9.726124
1,1,57,10.421772
2,2,57,10.846144
3,3,57,10.601064
4,4,51,11.463249
...,...,...,...
91,91,10,13.035835
92,92,10,13.403407
93,93,10,14.927353
94,94,10,14.641180


In [13]:
display(feature_horizon_matrix[:20])

,feature,h1,h2,h3,h4,h5,h6,h7,h8,h9,...,h87,h88,h89,h90,h91,h92,h93,h94,h95,h96
0,nsw_price_asinh_lag_1,False,False,False,False,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
1,nsw_price_asinh_lag_2,False,False,False,False,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
2,nsw_price_asinh_lag_4,False,False,False,False,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
3,nsw_price_asinh_lag_12,False,False,False,False,False,True,True,True,False,...,False,False,False,False,True,False,False,False,False,False
4,nsw_price_asinh_lag_48,True,True,True,True,False,True,True,True,False,...,False,False,False,False,True,False,False,False,False,False
5,nsw_price_asinh_lag_96,True,True,True,True,True,True,True,True,False,...,False,False,False,False,True,False,False,False,False,False
6,nsw_price_asinh_lag_336,True,True,True,True,True,True,True,True,False,...,False,False,False,False,True,False,False,False,False,False
7,nsw_price_asinh_lag_337,True,True,True,True,True,True,True,True,False,...,False,False,False,False,True,False,False,False,False,False
8,nsw_price_asinh_rmean_48,False,False,False,False,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
9,nsw_price_asinh_rmean_336,True,True,True,False,False,True,True,True,False,...,False,False,False,False,True,False,False,False,False,False


Export

In [14]:
feature_horizon_matrix.to_parquet(variables.FEATURES_OPTIMAL_AMOUNT_PATH)